# **Problem 5**

You will need to complete this problem on an alternative platform, such as Google Colab. Once you are finished, run all cells and show all outputs. You can download the notebook as a PDF and append it to the rest of your homework submission.

This notebook builds a vision transformer for image classification.  It works with the [CIFAR10](https://www.cs.toronto.edu/~kriz/cifar.html) dataset, which consists of 60k 32x32 color images in 10 classes.

If you are using Google Colab, you can change your runtime to an instance with GPU support to speed up training, e.g. a T4 GPU.

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import numpy as np
import random
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [34]:
# Hyperparameters
BATCH_SIZE = 128
PATCH_SIZE = 4
NUM_CLASSES = 10
IMAGE_SIZE = 32
CHANNELS = 3
EMBED_DIM = 256
NUM_HEADS = 8
DEPTH = 6
MLP_DIM = 512
DROP_RATE = 0.1

In [35]:
# Image transformations
transform = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                transforms.RandomHorizontalFlip(),
                                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
                                transforms.ToTensor(),
                                transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5))
                                ])

# Get datasets and convert to dataloaders
train_loader = DataLoader(
    datasets.CIFAR10(root='data', train=True, download=True, transform=transform),
    batch_size=BATCH_SIZE, shuffle=True)

test_loader = DataLoader(
    datasets.CIFAR10(root='data', train=False, download=True, transform=transform),
    batch_size=BATCH_SIZE, shuffle=False)

In [36]:
# Let's draw some of the training data

def plot_grid(dataset, classes, grid_size=3):
    fig, axes = plt.subplots(grid_size, grid_size, figsize=(5, 5))
    for i in range(grid_size):
        for j in range(grid_size):
            idx = random.randint(0, len(dataset) - 1)
            img = dataset[idx]
            input_tensor = img.unsqueeze(dim=0).to(device)
            with torch.inference_mode():
                img = img * 0.5 + 0.5 # unnormalize image
                npimg = img.cpu().numpy()
                axes[i, j].imshow(np.transpose(npimg, (1, 2, 0)))
                axes[i, j].set_title(f"Class: {cifar_classes[classes[idx]]}", fontsize=10)
                axes[i, j].axis("off")
    plt.tight_layout()
    plt.show()

examples = enumerate(test_loader)
cifar_classes = test_loader.dataset.classes
batch_idx, (example_data, example_targets) = next(examples)
plot_grid(example_data, example_targets)

## Part 1 (8 points)

Define the neural network by completing the `MLP` and `TransformerEncoderLayer` classes below. These two, along with `PatchEmbedding`, will be used to define the overall `VisionTransformer`.

In [37]:
# TODO Implement a multi-layer perceptron.
# In __init__(), define:
# 1. A linear layer (fc1) mapping in_features to hidden_features.
# 2. A linear layer (fc2) mapping hidden_features to in_features.
# 3. A dropout layer with drop_rate.

# In forward(), process the input x through:
# 1. fc1, followed by ReLU activation and dropout.
# 2. fc2, followed by dropout.

class MLP(nn.Module):
    def __init__(self, in_features, hidden_features, drop_rate):
        super().__init__()

    def forward(self, x):
        return x

In [38]:
# TODO Implement the TransformerEncoderLayer.
# In __init__(), define:
# 1. Two LayerNorm layers (norm1, norm2), each with embed_dim.
# 2. A MultiHeadAttention layer (attn), given embed_dim, num_heads, and drop_rate (batch_first=True).
# 3. An MLP layer (mlp), given embed_dim, mlp_dim, and drop_rate.

# In forward(), process x as follows, ensuring residual connections are added after each sub-layer:
# 1. Normalize x using norm1.
# 2. Apply attn to the normalized x (using it as query, key, and value). Remember to extract the attention output.
# 3. Add the attention output to the original x (residual connection).
# 4. Normalize the result using norm2.
# 5. Apply mlp to the normalized result.
# 6. Add the MLP output to the input of the mlp (residual connection).

class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, drop_rate):
        super().__init__()

    def forward(self, x):
        return x

In [39]:
class PatchEmbedding(nn.Module):
  def __init__(self, img_size, patch_size, in_channels, embed_dim):
      super().__init__()
      self.patch_size = patch_size
      self.proj = nn.Conv2d(in_channels=in_channels,
                            out_channels=embed_dim,
                            kernel_size=patch_size,
                            stride=patch_size)
      num_patches = (img_size // patch_size) ** 2
      self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
      self.pos_embed = nn.Parameter(torch.randn(1, 1 + num_patches, embed_dim))


  def forward(self, x):
      B = x.size(0)
      x = self.proj(x) # (B, E, H/P, W/P)
      x = x.flatten(2).transpose(1, 2) # (B, N, E)
      cls_token = self.cls_token.expand(B, -1 , -1)
      x = torch.cat((cls_token, x), dim=1)
      x = x + self.pos_embed
      return x

In [40]:
class VisionTransformer(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, num_classes, embed_dim, depth, num_heads, mlp_dim, drop_rate):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        self.encoder = nn.Sequential(*[
            TransformerEncoderLayer(embed_dim, num_heads, mlp_dim, drop_rate)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

  def forward(self, x):
      x = self.patch_embed(x)
      x = self.encoder(x)
      x = self.norm(x)
      cls_token = x[:, 0]
      return self.head(cls_token)

## Part 2 (6 points)

Write the main training routine. It is a function that trains the `model` on the data `loader`, using the given `optimizer` and `criterion`. It should return the average loss and classification accuracy.

In [41]:
# TODO Implement the main training routine
# Return the average loss and classification accuracy

def train(model, loader, optimizer, criterion):
    pass

In [42]:
def evaluate(model, loader, criterion):
    model.eval()
    test_loss = 0
    correct = 0

    with torch.no_grad():
        for data, target in loader:
            data = data.to(device)
            target = target.to(device)
            output = model(data)

            test_loss += criterion(output, target).item() * data.size(0)
            pred = output.data.max(1, keepdim=True)[1]
            correct += (pred.squeeze() == target).sum().item()

    return test_loss / len(loader.dataset), correct / len(loader.dataset)

## Part 3 (6 points)

Define an instance of the vision transformer network using the defined hyperparameters at the beginning. Define an Adam optimizer with a learning rate of 0.0003, and a criterion using cross entropy loss.

With everything defined, train the model for 20 epochs. Generate two plots, one showing the training and test losses, and one showing the training and test accuracies, both as a function of epoch.

